<a href="https://colab.research.google.com/github/Prktriesitout/LangGraph/blob/main/Langgraph6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Short-Term and Long-Term Memory Chat with OpenAI

A chatbot feels natural only if it remembers what was just said and what it knows about the user.


Both live in the state , and MemorySaver checkpoints the whole state after every turn . Each new turn on the same tread_id therefore starts with the earlier messages and the profile

start -> chat_node -> recall_node -> END

chat_node reads the latest user message, extracts any facts (name, preferences) and merges them into user_profile.
recall_node reads user_profile and the chat history, then writes a personalised reply.

In [1]:
!pip install -q langgraph langchain-core langchain-openai

In [3]:
import json
import re
from typing import TypedDict, Annotated

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import AIMessage, SystemMessage
from langchain_openai import ChatOpenAI


In [4]:
class State(TypedDict):
  messages: Annotated[list, add_messages]
  user_profile: dict



In [5]:
llm = ChatOpenAI(model="gpt-4o-mini",temperature=0)
def text_of(msg) -> str:
  c = msg.content
  if isinstance(c, str):
    return c.strip()
  return "".join(b.get("text", "") for b in c if isinstance(b, dict)).strip()

In [6]:
def parse_json(text: str) -> dict:
    """Pull the first {...} out of the reply, tolerating code fences."""
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return {}
    try:
        data = json.loads(match.group(0))
        return data if isinstance(data, dict) else {}
    except json.JSONDecodeError:
        return {}

In [8]:
def chat_node(state: State) -> dict:
  user_msg = state["messages"][-1].content
  reply = llm.invoke(
      "Extract facts the user states about themselves from the message below. "
        "Reply with ONLY a JSON object using short snake_case keys, for example "
        '{"name": "Ravi", "favorite_food": "pizza"}. '
        "If there are no personal facts, reply with {}.\n\n"
        f"Message: {user_msg}"
  )
  new_facts = parse_json(text_of(reply))

  profile = {**state.get("user_profile", {}), **new_facts}
  print(f"[chat_node] profile now:{profile}")
  return {"user_profile": profile}


In [9]:
def recall_node(state: State) -> dict:
    profile = state.get("user_profile", {})
    system = SystemMessage(content=(
        "You are a friendly assistant. Known facts about the user: "
        f"{json.dumps(profile)}. Use them naturally to personalise your reply, "
        "and use the conversation so far for context. Keep it to 1-2 sentences."
    ))
    reply = llm.invoke([system] + state["messages"])
    return {"messages": [AIMessage(content=text_of(reply))]}

In [10]:
builder = StateGraph(State)
builder.add_node("chat_node", chat_node)
builder.add_node("recall_node", recall_node)

builder.add_edge(START, "chat_node")
builder.add_edge("chat_node", "recall_node")
builder.add_edge("recall_node", END)

graph = builder.compile(checkpointer=MemorySaver())   # short-term memory

In [11]:
config = {"configurable": {"thread_id": "ravi-chat"}}

def say(text):
    print(f"\nUser: {text}")
    out = graph.invoke({"messages": [("user", text)]}, config)
    print(f"Bot: {out['messages'][-1].content}")

say("Hi, my name is Ravi and I love pizza.")
say("I'm preparing for a math exam next week.")
say("What's my name and what food do I like?")        # long-term facts
say("What was the first thing I told you?")           # short-term history


User: Hi, my name is Ravi and I love pizza.
[chat_node] profile now:{'name': 'Ravi', 'favorite_food': 'pizza'}
Bot: Hi Ravi! It's great to meet you! Pizza is such a delicious choice—do you have a favorite topping?

User: I'm preparing for a math exam next week.
[chat_node] profile now:{'name': 'Ravi', 'favorite_food': 'pizza'}
Bot: That sounds important, Ravi! If you need any help with math concepts or tips for studying, feel free to ask. And maybe reward yourself with some pizza after the exam!

User: What's my name and what food do I like?
[chat_node] profile now:{'name': 'Ravi', 'favorite_food': 'pizza'}
Bot: Your name is Ravi, and you love pizza!

User: What was the first thing I told you?
[chat_node] profile now:{'name': 'Ravi', 'favorite_food': 'pizza'}
Bot: You introduced yourself by saying your name is Ravi and that you love pizza.


In [12]:
snapshot = graph.get_state(config)
print("user_profile:", snapshot.values["user_profile"])
print("messages stored:", len(snapshot.values["messages"]))

user_profile: {'name': 'Ravi', 'favorite_food': 'pizza'}
messages stored: 8
